In [ ]:
%load_ext autoreload

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

In [ ]:
%autoreload 2

import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from PIL import Image
from rasterio.mask import mask

from estuary.util.img import false_color, normalized_image_3_channel

In [ ]:
dove_labels = pd.read_csv("/Users/kyledorman/data/estuary/dataset/train.csv")
dove_labels["acquired"] = pd.to_datetime(dove_labels["acquired"], errors="coerce", utc=True)
dove_labels = (
    dove_labels[dove_labels.label != "unsure"]
    .reset_index(drop=True)
    .sort_values(["region", "acquired"])
)
ss_labels = pd.read_csv("/Volumes/x10pro/estuary/labeling/skysat/labels.csv")
ss_labels["acquired"] = pd.to_datetime(ss_labels["acquired"], errors="coerce", utc=True)
ss_labels = (
    ss_labels[ss_labels.label != "unsure"]
    .reset_index(drop=True)
    .sort_values(["region", "acquired"])
)

tol = pd.Timedelta("10h")

# cross-join within region, then filter by window
tmp = dove_labels.merge(ss_labels, on="region", suffixes=("_dd", "_ss"))
mask = (tmp["acquired_dd"] >= tmp["acquired_ss"] - tol) & (
    tmp["acquired_dd"] <= tmp["acquired_ss"] + tol
)
pairs = tmp.loc[mask].sort_values(["region", "acquired_dd"])

pairs.head()

In [ ]:
region_df = pairs[
    # (pairs.label_ss != pairs.label_dd)
    pairs.region == 28
][
    [
        "region",
        "source_tif_dd",
        "label_dd",
        "acquired_dd",
        "source_tif_ss",
        "label_ss",
        "acquired_ss",
    ]
].reset_index(drop=True)
region_df["time_diff"] = (region_df.acquired_dd - region_df.acquired_ss).abs()
region_df["label_match"] = region_df.label_dd == region_df.label_ss
region_df.head(10)

In [ ]:
plt.figure(figsize=(8, 8))
row = region_df.iloc[0]
print(row.acquired_dd)
with rasterio.open(row.source_tif_ss) as src:
    data = src.read()
    nodata = src.read(1, masked=True).mask
    img = false_color(data, nodata)
    img = Image.fromarray(img)
    # .save("/Users/kyledorman/data/estuary/display/region_53.png")
plt.imshow(img)

In [ ]:
region_df[["acquired_dd", "label_dd", "label_ss"]]

In [ ]:
to_eval = [
    (21, datetime.datetime(year=2017, month=12, day=18)),
    (21, datetime.datetime(year=2019, month=4, day=12)),
    (21, datetime.datetime(year=2021, month=5, day=3)),
    (28, datetime.datetime(year=2023, month=11, day=12)),
]

In [ ]:
def dove_from_date(region, date):
    base = Path("/Volumes/x10pro/estuary/sat_data/")
    pth = list(
        base.glob(
            f"*ove/results/{date.year}/{date.month}/{region}/files/{date.year}{date.month:02}{date.day:02}*_SR*clip.tif"
        )
    )[0]
    print("dove", pth.stem)
    with rasterio.open(pth) as src:
        data = src.read(out_dtype=np.float32)
        nodata = src.read(1, masked=True).mask
        img = false_color(data, nodata)

    nir = data[3]
    g = data[1]
    num = g - nir
    denom = g + nir
    wi = num / denom.clip(1, denom.max())

    return img, wi


img, wi = dove_from_date(28, datetime.datetime(year=2023, month=10, day=22))

plt.figure(figsize=(10, 10))
plt.imshow(img)
# plt.figure()
# plt.imshow(wi)

In [ ]:
def s2_false_color(bands):
    nodata = (bands == 0).all(axis=0)
    img = normalized_image_3_channel(bands, nodata, (7, 3, 2)).transpose((1, 2, 0))
    k = 1.5
    img = np.tanh(k * img) / np.tanh(k)
    img = np.array(img * 255, dtype=np.uint8)
    return img


def s2_mdwi(s2_dir):
    band_names = ["B03", "B11"]
    bands = []
    for name in band_names:
        file = list(s2_dir.glob(f"*_{name}_*"))[0]
        with rasterio.open(file) as src:
            bands.append(src.read(1, out_dtype=np.float32))

    denom = bands[0] + bands[1]
    num = bands[0] - bands[1]

    return num / denom.clip(1, denom.max())


def s2_awei_img(s2_dir):
    bands = []
    for name in S2_BAND_NAMES:
        file = list(s2_dir.glob(f"*_{name}_*"))[0]
        with rasterio.open(file) as src:
            bands.append(src.read(1, out_dtype=np.float32))

    green = bands[2]
    NIR = bands[7]
    SWIR = bands[10]
    MIR = bands[11]
    AWEI = (4 * (green - MIR)) - (0.25 * NIR - 2.75 * SWIR)
    return AWEI
    # return contrast_stretch(np.log10(1 + AWEI - AWEI.min()))


S2_BAND_NAMES = [
    "B01",
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B8A",
    "B09",
    "B11",
    "B12",
]


def s2_img(s2_dir):
    bands = []
    for name in S2_BAND_NAMES:
        file = list(s2_dir.glob(f"*_{name}_*"))[0]
        with rasterio.open(file) as src:
            bands.append(src.read(1, out_dtype=np.float32))

    return s2_false_color(np.array(bands))


def s2_img_from_date(date):
    pth = Path(
        f"/Users/kyledorman/data/estuary/sat_compare/sentinel/s2-{date.month:02}-{date.day:02}-{date.year}/"
    )
    print("s2", pth.stem)
    img = s2_img(pth)
    wi = s2_mdwi(pth)
    return img, wi


dt = to_eval[-1][1]
img, wi = s2_img_from_date(dt)
plt.figure(figsize=(10, 10))
plt.imshow(img)

In [ ]:
def skysat_from_date(region, date):
    base = Path(f"/Volumes/x10pro/estuary/sat_data/skysat/results/{date.year}/{region}/files/")
    pth = list(base.glob(f"{date.year}{date.month:02}{date.day:02}*_pansharpened_clip.tif"))[0]
    print("skysat", pth.stem)
    with rasterio.open(pth) as src:
        data = src.read()
        nodata = src.read(1, masked=True).mask
        img = false_color(data, nodata)

    nir = data[3]
    g = data[1]
    num = g - nir
    denom = g + nir
    wi = num / denom.clip(1, denom.max())

    return img, wi

In [ ]:
def plot_sats(region, date, save_dir: Path | None = None):
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 6))

    s2i, s2w = s2_img_from_date(date)
    dovei, dovew = dove_from_date(region, date)
    ssi, ssw = skysat_from_date(region, date)

    axes[0].axis("off")
    axes[0].imshow(ssi)
    axes[0].set_title("SkySat (0.5m)")

    axes[1].axis("off")
    axes[1].imshow(dovei)
    axes[1].set_title("Dove (3m)")

    axes[2].axis("off")
    axes[2].imshow(s2i)
    axes[2].set_title("Sentinel 2 (10m)")

    fig.tight_layout()
    if save_dir is not None:
        save_dir.mkdir(exist_ok=True, parents=True)
        save_path = save_dir / f"{region}_{date.year}_{date.month:02}_{date.day:02}.png"
        plt.savefig(save_path, dpi=300)
    plt.show()

In [ ]:
to_eval

In [ ]:
plot_sats(*to_eval[0], Path("/Users/kyledorman/data/estuary/stats_presentation"))

In [ ]:
plot_sats(*to_eval[1], Path("/Users/kyledorman/data/estuary/stats_presentation"))

In [ ]:
plot_sats(*to_eval[2], Path("/Users/kyledorman/data/estuary/stats_presentation"))

In [ ]:
plot_sats(*to_eval[3], Path("/Users/kyledorman/data/estuary/stats_presentation"))